In [4]:
import pandas as pd
import os
import glob

# Define paths
job_path = "data/raw/jobs/*.txt"
syllabus_path = "data/raw/syllabus_*.txt"

# Load all job descriptions
job_files = glob.glob(job_path)
job_data = []

for file in job_files:
    with open(file, 'r', encoding='utf-8') as f:
        text = f.read()
        filename = os.path.basename(file).replace('.txt', '')
        job_data.append({
            'filename': filename,
            'text': text,
            'type': 'job'
        })

# Load all syllabi
syllabus_files = glob.glob(syllabus_path)
syllabus_data = []

for file in syllabus_files:
    with open(file, 'r', encoding='utf-8') as f:
        text = f.read()
        filename = os.path.basename(file).replace('.txt', '')
        syllabus_data.append({
            'filename': filename,
            'text': text,
            'type': 'syllabus'
        })

# Combine into one DataFrame
all_data = pd.DataFrame(job_data + syllabus_data)

# Show what we loaded
print(f"✅ Loaded {len(all_data)} documents")
display(all_data.head())

✅ Loaded 42 documents


,filename,text,type
0,job_mobile_1,Job Title: Flutter Mobile Developer (Entry Lev...,job
1,job_cybersecurity_1,Full job description\n\nThe Cyber Security Spe...,job
2,job_cybersecurity_5,Job Title: Junior Penetration Tester (Ethical ...,job
3,job_blockchain_3,Job Title: Smart Contract Developer (Junior)\n...,job
4,job_blockchain_2,Blockchain Developer\n\nRole and Responsibilit...,job


In [5]:
import re

def clean_text(text):
    """
    Clean the text by:
    1. Converting to lowercase
    2. Removing punctuation and numbers
    3. Removing extra whitespace
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation and numbers (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning to all documents
all_data['cleaned_text'] = all_data['text'].apply(clean_text)

# Show the cleaned text side by side with the original
print("📊 Original vs Cleaned Text (First 3 rows):")
display(all_data[['filename', 'text', 'cleaned_text']].head(3))

📊 Original vs Cleaned Text (First 3 rows):


,filename,text,cleaned_text
0,job_mobile_1,Job Title: Flutter Mobile Developer (Entry Lev...,job title flutter mobile developer entry level...
1,job_cybersecurity_1,Full job description\n\nThe Cyber Security Spe...,full job description the cyber security specia...
2,job_cybersecurity_5,Job Title: Junior Penetration Tester (Ethical ...,job title junior penetration tester ethical ha...


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a TF-IDF Vectorizer
# max_features=1000 means we'll keep the top 1000 most important words
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')

# Convert all cleaned text into numbers (a matrix)
tfidf_matrix = vectorizer.fit_transform(all_data['cleaned_text'])

# Show what we created
print(f"✅ TF-IDF Matrix created!")
print(f"   📊 Shape: {tfidf_matrix.shape}")
print(f"   📝 Number of documents: {tfidf_matrix.shape[0]}")
print(f"   🔤 Number of unique words (features): {tfidf_matrix.shape[1]}")
print(f"\n📋 Here are the top 10 most important words:")
feature_names = vectorizer.get_feature_names_out()
print(feature_names[:10])

✅ TF-IDF Matrix created!
   📊 Shape: (42, 1000)
   📝 Number of documents: 42
   🔤 Number of unique words (features): 1000

📋 Here are the top 10 most important words:
['ability' 'academy' 'accelerating' 'acceptance' 'accepted' 'access'
 'accordance' 'according' 'accountability' 'accuracy']


In [7]:
# Let's see what the numbers look like for the first document
print("📊 TF-IDF values for the first document (job_mobile_1):")
print("Showing only words that have a value > 0:")
first_doc = tfidf_matrix[0].toarray()[0]
for i, value in enumerate(first_doc):
    if value > 0:
        print(f"   {feature_names[i]}: {value:.4f}")

📊 TF-IDF values for the first document (job_mobile_1):
Showing only words that have a value > 0:
   ability: 0.0730
   accepted: 0.1159
   agile: 0.0898
   android: 0.0961
   apis: 0.0585
   app: 0.3056
   apple: 0.0961
   applications: 0.0879
   authentication: 0.0898
   bugs: 0.0846
   build: 0.0978
   clean: 0.0961
   cloud: 0.0605
   code: 0.0626
   company: 0.0407
   control: 0.0730
   coursework: 0.1043
   crossplatform: 0.0898
   dart: 0.1692
   data: 0.1035
   debug: 0.0961
   deployment: 0.0764
   description: 0.0387
   design: 0.1427
   developer: 0.1346
   entry: 0.0802
   environment: 0.0730
   experience: 0.0352
   familiarity: 0.0566
   firebase: 0.1604
   firestore: 0.2087
   flutter: 0.3384
   git: 0.0700
   google: 0.0764
   handle: 0.0898
   highperformance: 0.0961
   implement: 0.0503
   interfaces: 0.1043
   intuitive: 0.1159
   ios: 0.0961
   issues: 0.0700
   job: 0.0794
   key: 0.0387
   knowledge: 0.0397
   level: 0.0764
   lifecycle: 0.0846
   looking: 0.0489
 

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# Separate syllabi and jobs
syllabi_indices = all_data[all_data['type'] == 'syllabus'].index
job_indices = all_data[all_data['type'] == 'job'].index

# Get the filenames for easier reading
syllabi_names = all_data.loc[syllabi_indices, 'filename'].values
job_names = all_data.loc[job_indices, 'filename'].values

# Calculate cosine similarity between all syllabi and all jobs
similarity_matrix = cosine_similarity(tfidf_matrix[syllabi_indices], tfidf_matrix[job_indices])

# Create a nice DataFrame to display results
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=syllabi_names,
    columns=job_names
)

print("📊 Similarity Matrix (Syllabi vs Jobs):")
print("   Higher numbers = More similar")
display(similarity_df.round(3))

📊 Similarity Matrix (Syllabi vs Jobs):
   Higher numbers = More similar


,job_mobile_1,job_cybersecurity_1,job_cybersecurity_5,job_blockchain_3,job_blockchain_2,job_blockchain_4,job_cybersecurity_4,job_data_eng_4,job_qa_1,job_qa_3,...,job_system_integration_1,job_qa_2,job_mobile_5,job_qa_5,job_system_integration_3,job_data_eng_5,job_data_eng_2,job_data_eng_3,job_cloud_5,job_cloud_1
syllabus_mobile,0.456,0.017,0.015,0.019,0.020,0.0,0.005,0.030,0.062,0.044,...,0.073,0.062,0.327,0.048,0.075,0.017,0.002,0.030,0.078,0.075
syllabus_data_eng,0.073,0.021,0.013,0.006,0.058,0.0,0.029,0.297,0.008,0.017,...,0.119,0.055,0.049,0.047,0.060,0.349,0.321,0.423,0.189,0.109
syllabus_system_integration,0.069,0.046,0.021,0.015,0.032,0.0,0.014,0.063,0.039,0.059,...,0.289,0.007,0.065,0.037,0.249,0.121,0.046,0.076,0.045,0.039
syllabus_qa,0.012,0.025,0.040,0.024,0.006,0.0,0.028,0.028,0.399,0.342,...,0.018,0.322,0.039,0.379,0.007,0.006,0.014,0.021,0.031,0.032
syllabus_blockchain,0.009,0.014,0.032,0.481,0.229,0.0,0.009,0.014,0.049,0.050,...,0.033,0.032,0.039,0.039,0.029,0.017,0.012,0.024,0.013,0.059
syllabus_cybersecurity,0.016,0.131,0.189,0.053,0.044,0.0,0.108,0.021,0.101,0.093,...,0.045,0.117,0.041,0.113,0.013,0.015,0.000,0.008,0.055,0.044
syllabus_cloud,0.067,0.142,0.058,0.038,0.022,0.0,0.063,0.044,0.012,0.030,...,0.019,0.041,0.061,0.009,0.036,0.003,0.012,0.096,0.260,0.404


In [9]:
# For each syllabus, find the top 3 most similar jobs
print("🎯 Top 3 Job Matches for Each Elective:")
print("=" * 60)

for syllabus in syllabi_names:
    # Get similarity scores for this syllabus
    scores = similarity_df.loc[syllabus]
    
    # Sort from highest to lowest
    top_jobs = scores.sort_values(ascending=False).head(3)
    
    # Print results
    print(f"\n📚 {syllabus.replace('syllabus_', '').upper()}:")
    for job, score in top_jobs.items():
        print(f"   → {job.replace('job_', '')}: {score:.3f}")

🎯 Top 3 Job Matches for Each Elective:

📚 MOBILE:
   → mobile_3: 0.549
   → mobile_1: 0.456
   → mobile_2: 0.343

📚 DATA_ENG:
   → data_eng_1: 0.475
   → data_eng_3: 0.423
   → data_eng_5: 0.349

📚 SYSTEM_INTEGRATION:
   → system_integration_1: 0.289
   → system_integration_3: 0.249
   → system_integration_2: 0.199

📚 QA:
   → qa_1: 0.399
   → qa_5: 0.379
   → qa_3: 0.342

📚 BLOCKCHAIN:
   → blockchain_3: 0.481
   → blockchain_2: 0.229
   → blockchain_1: 0.118

📚 CYBERSECURITY:
   → cybersecurity_3: 0.236
   → cybersecurity_2: 0.206
   → cybersecurity_5: 0.189

📚 CLOUD:
   → cloud_1: 0.404
   → cloud_4: 0.321
   → cloud_3: 0.266


In [10]:
# For each job, find the most similar syllabus
print("🏆 Best Elective for Each Job Role:")
print("=" * 60)

for job in job_names[:5]:  # Show first 5 jobs
    # Get similarity scores for this job
    scores = similarity_df[job]
    
    # Find the best matching syllabus
    best_syllabus = scores.sort_values(ascending=False).index[0]
    best_score = scores.sort_values(ascending=False).values[0]
    
    print(f"\n💼 {job.replace('job_', '')}:")
    print(f"   → Best elective: {best_syllabus.replace('syllabus_', '')} ({best_score:.3f})")

🏆 Best Elective for Each Job Role:

💼 mobile_1:
   → Best elective: mobile (0.456)

💼 cybersecurity_1:
   → Best elective: cloud (0.142)

💼 cybersecurity_5:
   → Best elective: cybersecurity (0.189)

💼 blockchain_3:
   → Best elective: blockchain (0.481)

💼 blockchain_2:
   → Best elective: blockchain (0.229)
